In [2]:
from pathlib import Path

DATA_DIR = Path("/home/danila/networks/data")
TEXT_DIR = DATA_DIR / "text"          # data/text/00000.txt ...
TRAIN_CSV = DATA_DIR / "train_split.csv"
VALID_CSV = DATA_DIR / "valid_split.csv"

MODEL_NAME_TEXT = "intfloat/e5-large-v2"  # high-quality embeddings
ID_WIDTH = 5

OUT_DIR = DATA_DIR / "embeddings" / "text_e5_large_v2_global_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = OUT_DIR / "index.parquet"

# runtime
DEVICE = "cuda"
BATCH_SIZE = 64
MAX_LENGTH = 512
CHUNK_TOKENS = 480

# saving
EMB_SAVE_DTYPE = "float16"
SKIP_IF_EXISTS = True

In [3]:
import json
import traceback
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

def _ts():
    return datetime.now().strftime("%H:%M:%S")

torch.set_grad_enabled(False)
print("CUDA:", torch.cuda.is_available())

CUDA: True


In [4]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})

train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)

all_ids = sorted(pd.concat([train_df["Filename"], valid_df["Filename"]]).unique().tolist())
tqdm.write(f"[{_ts()}] Total ids: {len(all_ids)}")
tqdm.write(f"Examples: {all_ids[:10]}")

[19:43:52] Total ids: 12660
Examples: ['00000', '00001', '00002', '00003', '00004', '00005', '00006', '00007', '00008', '00009']


In [5]:
assert torch.cuda.is_available() or DEVICE == "cpu", "CUDA not available; set DEVICE='cpu' if needed."

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_TEXT, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME_TEXT)
model.eval().to(DEVICE)

# hidden size
HIDDEN = int(model.config.hidden_size)
tqdm.write(f"[{_ts()}] Loaded {MODEL_NAME_TEXT} | hidden={HIDDEN} | device={DEVICE}")

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

[19:44:26] Loaded intfloat/e5-large-v2 | hidden=1024 | device=cuda


In [11]:
import re

def read_text(video_id: str) -> str:
    path = TEXT_DIR / f"{video_id}.txt"
    if not path.exists():
        raise FileNotFoundError(str(path))
    txt = path.read_text(encoding="utf-8", errors="ignore")
    # normalization of spaces
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

def chunk_text_by_tokens_with_lengths(text: str, chunk_tokens: int):
    """
    returns:
      chunks: list[str]
      lengths: list[int]  # number of tokens (without special tokens) in each chunk
    """
    import re
    if not text:
        return [""], [0]

    ids = tokenizer(
        text,
        add_special_tokens=False,
        return_attention_mask=False,
        return_token_type_ids=False
    )["input_ids"]

    if len(ids) <= chunk_tokens:
        chunk = re.sub(r"\s+", " ", text).strip()
        return [chunk], [len(ids)]

    chunks, lengths = [], []
    for i in range(0, len(ids), chunk_tokens):
        sub = ids[i:i+chunk_tokens]
        chunk = tokenizer.decode(sub, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        chunk = re.sub(r"\s+", " ", chunk).strip()
        if chunk:
            chunks.append(chunk)
            lengths.append(len(sub))

    if not chunks:
        chunk = re.sub(r"\s+", " ", text).strip()
        return [chunk], [len(ids)]

    return chunks, lengths

In [7]:
def mean_pooling(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    # last_hidden_state: [B, L, H], attention_mask: [B, L]
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)  # [B,L,1]
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp_min(1e-9)
    return summed / counts

def l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return x / (x.norm(p=2, dim=-1, keepdim=True).clamp_min(eps))

In [8]:
@torch.inference_mode()
def encode_passages(passages: list[str], batch_size: int = 64, max_length: int = 512) -> np.ndarray:
    """
    passages: list[str]
    returns: np.ndarray [N, H] float32
    """
    embs = []
    for batch in range(0, len(passages), batch_size):
        texts = passages[batch:batch+batch_size]
        # E5: "passage: {text}"
        texts = [f"passage: {t}" for t in texts]

        tok = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        tok = {k: v.to(DEVICE) for k, v in tok.items()}

        out = model(**tok, return_dict=True)
        pooled = mean_pooling(out.last_hidden_state, tok["attention_mask"])
        pooled = l2_normalize(pooled)
        embs.append(pooled.detach().cpu().float().numpy())

    return np.concatenate(embs, axis=0) if embs else np.zeros((0, HIDDEN), dtype=np.float32)

In [17]:
def process_one_text(video_id: str):
    txt_path = TEXT_DIR / f"{video_id}.txt"
    if not txt_path.exists():
        return None, {"video_id": video_id, "reason": f"txt_not_found: {txt_path}"}

    out_npz  = OUT_DIR / f"{video_id}.npz"
    out_meta = OUT_DIR / f"{video_id}.json"

    if SKIP_IF_EXISTS and out_npz.exists() and out_meta.exists():
        try:
            d = json.loads(out_meta.read_text())
            d["video_id"] = d.get("video_id", video_id)
            d["npz_path"] = str(out_npz)
            return d, None
        except Exception:
            return {"video_id": video_id, "npz_path": str(out_npz), "skipped": True}, None

    text = read_text(video_id)
    chunks, lengths = chunk_text_by_tokens_with_lengths(text, CHUNK_TOKENS)
    
    chunk_embs = encode_passages(chunks, batch_size=BATCH_SIZE, max_length=MAX_LENGTH)  # [N,H]
    
    if chunk_embs.shape[0] == 0:
        emb = np.zeros((HIDDEN,), dtype=np.float32)
        num_chunks = 0
    else:
        w = np.asarray(lengths[:chunk_embs.shape[0]], dtype=np.float32)
        if w.sum() <= 0:
            w = np.ones_like(w, dtype=np.float32)
    
        w = w / (w.sum() + 1e-12)  # we normalize weights
    
        # weighted average for chunksweighted average for chunks
        emb = (chunk_embs * w[:, None]).sum(axis=0)
    
        # final normalization
        emb = emb / (np.linalg.norm(emb) + 1e-12)
    
        num_chunks = int(chunk_embs.shape[0])

    if EMB_SAVE_DTYPE == "float16":
        emb_save = emb.astype(np.float16)
    else:
        emb_save = emb.astype(np.float32)

    meta = {
        "video_id": video_id,
        "txt_path": str(txt_path),
        "npz_path": str(out_npz),
        "model": MODEL_NAME_TEXT,
        "hidden_size": int(HIDDEN),
        "max_length": int(MAX_LENGTH),
        "chunk_tokens": int(CHUNK_TOKENS),
        "batch_size": int(BATCH_SIZE),
        "num_chunks": int(num_chunks),
        "text_len_chars": int(len(text)),
        "text_len_words": int(len(text.split())) if text else 0,
        "save_dtype": EMB_SAVE_DTYPE,
    }

    np.savez_compressed(out_npz, embedding=emb_save)
    out_meta.write_text(json.dumps(meta, ensure_ascii=False, indent=2))

    return meta, None

In [19]:
metas = []
failed = []

pbar = tqdm(all_ids, desc="Extract text embeddings (E5 global)")
total = len(all_ids)

for i, vid in enumerate(pbar, start=1):
    try:
        meta, err = process_one_text(vid)
        if err is not None:
            failed.append(err)
            tqdm.write(f"[{_ts()}] ERROR {vid}: {err['reason']}")
        else:
            metas.append(meta)
    except Exception as e:
        failed.append({"video_id": vid, "reason": repr(e)})
        tqdm.write(f"[{_ts()}] ERROR {vid}: {repr(e)}")
        tqdm.write(traceback.format_exc().rstrip())

    if i % 100 == 0:
        tqdm.write(f"[{_ts()}] progress {i}/{total} ok={len(metas)} failed={len(failed)}")

# save failed.csv
if len(failed):
    fail_path = OUT_DIR / "failed.csv"
    pd.DataFrame(failed).to_csv(fail_path, index=False)
    tqdm.write(f"[{_ts()}] Saved failures: {fail_path}")

Extract text embeddings (E5 global):   0%|          | 0/12660 [00:00<?, ?it/s]

[19:57:36] progress 100/12660 ok=100 failed=0
[19:57:38] progress 200/12660 ok=200 failed=0
[19:57:39] progress 300/12660 ok=300 failed=0
[19:57:41] progress 400/12660 ok=400 failed=0
[19:57:42] progress 500/12660 ok=500 failed=0
[19:57:44] progress 600/12660 ok=600 failed=0
[19:57:45] progress 700/12660 ok=700 failed=0
[19:57:47] progress 800/12660 ok=800 failed=0
[19:57:48] progress 900/12660 ok=900 failed=0
[19:57:50] progress 1000/12660 ok=1000 failed=0
[19:57:52] progress 1100/12660 ok=1100 failed=0
[19:57:53] progress 1200/12660 ok=1200 failed=0
[19:57:55] progress 1300/12660 ok=1300 failed=0
[19:57:56] progress 1400/12660 ok=1400 failed=0
[19:57:58] progress 1500/12660 ok=1500 failed=0
[19:57:59] progress 1600/12660 ok=1600 failed=0
[19:58:01] progress 1700/12660 ok=1700 failed=0
[19:58:02] progress 1800/12660 ok=1800 failed=0
[19:58:04] progress 1900/12660 ok=1900 failed=0
[19:58:06] progress 2000/12660 ok=2000 failed=0
[19:58:07] progress 2100/12660 ok=2100 failed=0
[19:58:09]

In [14]:
meta_rows = []
json_files = sorted(Path(OUT_DIR).glob("*.json"))

for jf in tqdm(json_files, desc="Load saved text metas"):
    try:
        d = json.loads(jf.read_text())
        vid = d.get("video_id", jf.stem)
        d["video_id"] = vid

        npz_path = Path(d.get("npz_path", OUT_DIR / f"{vid}.npz"))
        d["npz_path"] = str(npz_path)
        d["npz_exists"] = npz_path.exists()

        meta_rows.append(d)
    except Exception as e:
        tqdm.write(f"[{_ts()}] meta read error {jf.name}: {repr(e)}")

meta_df = pd.DataFrame(meta_rows)

fail_path = OUT_DIR / "failed.csv"
if fail_path.exists():
    fail_df = pd.read_csv(fail_path, dtype={"video_id": str})
else:
    fail_df = pd.DataFrame(columns=["video_id", "reason"])

meta_df.to_parquet(INDEX_PATH, index=False)
tqdm.write(f"[{_ts()}] Saved index: {INDEX_PATH}")
tqdm.write(f"[{_ts()}] DONE | ok={len(meta_df)} failed={len(fail_df)} metas_json={len(json_files)}")

Load saved text metas:   0%|          | 0/12660 [00:00<?, ?it/s]

[19:54:07] Saved index: /home/danila/networks/data/embeddings/text_e5_large_v2_global_v1/index.parquet
[19:54:07] DONE | ok=12660 failed=0 metas_json=12660
